In [1]:
from func import *

In [2]:
train_df = pd.read_csv("train_df_clean.csv")
test_df = pd.read_csv("test_df_clean.csv")

no_missing_train_df = train_df.dropna(axis=1)

no_missing_test_df = test_df.dropna(axis=1)

no_missing_train_df.drop(columns=["log_ClosePrice"], inplace=True)
no_missing_test_df.drop(columns=["log_ClosePrice"], inplace=True)

train_df = no_missing_train_df
test_df = no_missing_test_df

C:\Users\lee39\AppData\Local\Temp\ipykernel_5140\1588991985.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  no_missing_train_df.drop(columns=["log_ClosePrice"], inplace=True)
C:\Users\lee39\AppData\Local\Temp\ipykernel_5140\1588991985.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  no_missing_test_df.drop(columns=["log_ClosePrice"], inplace=True)


In [3]:
X_train = train_df.drop(columns=["ClosePrice"], axis=1)

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

nunique = X_train[cat_cols].nunique(dropna=False)

low_card_cols = nunique[(nunique <= 20)].index.tolist()
high_card_cols = nunique[(nunique >= 20)].index.tolist()


# Elastic Net

In [4]:
from sklearn.linear_model import ElasticNet

In [5]:

# Tuning
en_param_grid = param_grid = {
    "model__alpha": np.logspace(-4, 2, 25),
    "model__l1_ratio": np.linspace(0.05, 0.95, 19),
}

res = grid_tune_with_make_model_pipeline(
    train_df=train_df,
    target_col="ClosePrice",
    model=ElasticNet(max_iter=20000),
    param_grid=param_grid,
    col_drop_list=[],
    card_threshold=20,
    scoring="r2",
    cv=5
)

print(res["best_score"], res["best_params"])

Fitting 5 folds for each of 475 candidates, totalling 2375 fits
-0.00042025482821088643 {'model__alpha': 10.0, 'model__l1_ratio': 0.85}


In [6]:
alpha = 0.01778279410038923
l_1_ratio = 0.05
model = ElasticNet(alpha=alpha,
                   l1_ratio=l_1_ratio)

elastic_net_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=model,
    target_col="ClosePrice",
    card_threshold=20,
    log_transformed=False
)
el_metrics = elastic_net_result["Metrics"]
print(el_metrics)

{'R2(log)': -0.0027012360728344564, 'R2': -0.0027012360728344564, 'MAPE': 74.28250280316287, 'MdAPE': 51.0437860849879, 'RMSE': 926101.2813575964, 'MAE': 611642.5222020246, 'Bias(mean residual)': 52388.27450340004, 'APE_95pct': 229.54161545370434, 'APE_99pct': 374.53968299450685, 'APE_max': 562.7649145031482, 'Train_R2(log)': 0.9999999999997592, 'Test_R2(log)': -0.0027012360728344564, 'R2_gap': 1.0027012360725935}


# XGB

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

from xgboost import XGBRegressor
from scipy.stats import randint, uniform, loguniform

In [8]:

xgb_param_dist = {
    "model__max_depth": randint(3, 7),                    # 3–6
    "model__min_child_weight": randint(8, 31),           # 8–30
    "model__learning_rate": loguniform(0.01, 0.08),      # small LR
    "model__n_estimators": randint(400, 1400),
    "model__subsample": uniform(0.6, 0.3),               # 0.6–0.9
    "model__colsample_bytree": uniform(0.5, 0.4),        # 0.5–0.9
    "model__reg_lambda": loguniform(1.0, 50.0),          # strong L2
    "model__reg_alpha": loguniform(1e-4, 2.0),           # mild–moderate L1
}


model = make_model_numeric_only_pipeline(model=XGBRegressor(
    random_state=42,
    n_jobs=1),
num_cols=num_cols,
num_scaler=None)


xgb_rs = RandomizedSearchCV(
    model,
    param_distributions=xgb_param_dist,
    n_iter=40,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

xgb_rs.fit(train_df.drop("ClosePrice", axis=1), train_df["ClosePrice"])
print(xgb_rs.best_params_)

{'model__colsample_bytree': 0.7844598129752072, 'model__learning_rate': 0.05383345943137039, 'model__max_depth': 6, 'model__min_child_weight': 14, 'model__n_estimators': 1219, 'model__reg_alpha': 1.10973369349242, 'model__reg_lambda': 4.736558858366902, 'model__subsample': 0.755325405158244}


In [9]:
xgb_model = XGBRegressor(n_estimators=785,
                         learning_rate=0.07399059420398583,
                         max_depth=4,
                         subsample=0.8993221455146825,
                         colsample_bytree=0.8887128330883842,
                         min_child_weight=14,
                         reg_lambda=1.7848234036017752,
                         reg_alpha=0.11574364959605028)


xgb_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=xgb_model,
    target_col="ClosePrice",
    card_threshold=20,
    num_scaler=None,
    num_only=True,
    log_transformed=False
)

xgb_metrics = xgb_result["Metrics"]
print(xgb_metrics)

{'R2(log)': 0.8576031761439273, 'R2': 0.8576031761439273, 'MAPE': 16.15684948617441, 'MdAPE': 10.896675046929472, 'RMSE': 348997.93815456104, 'MAE': 182966.4842261011, 'Bias(mean residual)': 30176.279249249248, 'APE_95pct': 49.474257416979945, 'APE_99pct': 85.23692704469515, 'APE_max': 231.5806785714286, 'Train_R2(log)': 0.9034043145793214, 'Test_R2(log)': 0.8576031761439273, 'R2_gap': 0.04580113843539402}


# Summary

In [10]:
models_summary = {
    "Elastic Net": el_metrics,
    "XGBoost": xgb_metrics,
}

df = pd.DataFrame.from_dict(models_summary).transpose()
df

,R2(log),R2,MAPE,MdAPE,RMSE,MAE,Bias(mean residual),APE_95pct,APE_99pct,APE_max,Train_R2(log),Test_R2(log),R2_gap
Elastic Net,-0.002701,-0.002701,74.282503,51.043786,926101.281358,611642.522202,52388.274503,229.541615,374.539683,562.764915,1.000000,-0.002701,1.002701
XGBoost,0.857603,0.857603,16.156849,10.896675,348997.938155,182966.484226,30176.279249,49.474257,85.236927,231.580679,0.903404,0.857603,0.045801
